# Data sources, audit and cohort construction

**Purpose.** This notebook traces the audited chart and feature sources into the clean analysis cohorts and adds the supervisor-requested bridge from strict first-spell survival to the final fixed-window cohort.

**Principal inputs.** Preserved chart archive; validated raw-source audit; first-spell linkage files; `pre_model_analysis_ready_v1`; frozen grouped assignments; `supervisor_confirmed_update_v1`.

**Main outputs.** Source diagnostics, exact matching, cohort exclusions and the ID-level 1,357-to-1,014 reconciliation. No source file is modified.


In [ ]:
from pathlib import Path
import json
import math
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

# Find the repository from the current working directory so the notebook
# remains portable when the project folder is moved or opened in Colab.
current = Path.cwd().resolve()
candidates = [current, *current.parents]
ROOT = next((p for p in candidates if (p / "scripts/thesis_utils_v2.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from within the Spotify_clv project.")
sys.path.insert(0, str(ROOT / "scripts"))

from thesis_utils_v2 import (
    SEED, assert_close, assert_unique_tracks, blocked_predictor_hits,
    concise_table, environment_table, extract_spotify_id, load_json,
    read_chart_archive, record_values, regression_metrics,
    save_and_show_figure, setup_notebook,
)

ROOT, FIGURE_DIR, RECORD_DIR = setup_notebook("01_data_sources_audit_and_cohort.ipynb")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
display(environment_table())
print(f"Random seed: {SEED}")


## Data sources and provenance

The Global Weekly Top 200 archive supplies the longitudinal chart history. The 550k source adds audio, genre and lyrics fields; the 160k source provides a larger exact-ID robustness population with a narrower feature set. Licence labels below reproduce the saved dataset-page documentation and are not a new legal opinion. The chart source was approved for the non-commercial thesis, but the working specification itself remains provisional.


In [ ]:
chart_metadata = load_json(
    ROOT / "data/raw/charts/source_metadata/bartomiejczyewski_top200_global_2017_2021.json"
)
sources = pd.DataFrame([
    {
        "source": "Global Weekly Top 200 (Bartłomiej Czyżewski)",
        "saved_owner": chart_metadata["owner"],
        "saved_licence": chart_metadata["kaggle_licence"],
        "identifier": "Spotify track URL / extracted track ID",
        "analytical_role": "weekly ranks, streams and chart presence",
    },
    {
        "source": "550k Spotify Songs (Tüysüz)",
        "saved_owner": "Tüysüz (Kaggle dataset page)",
        "saved_licence": "CC BY-NC-SA 4.0 (saved project documentation)",
        "identifier": "Spotify track ID",
        "analytical_role": "primary audio, genre and lyric features",
    },
    {
        "source": "160k Spotify Tracks (Y. E. Ay)",
        "saved_owner": "Y. E. Ay (Kaggle dataset page)",
        "saved_licence": "CDLA-Sharing-1.0 (saved project documentation)",
        "identifier": "Spotify track ID",
        "analytical_role": "common-feature robustness cohort",
    },
])
display(sources)
print("Chart source URL:", chart_metadata["source_url"])
print("Preserved archive SHA-256:", chart_metadata["original_archive_sha256"])


## Direct audit of the chart archive

The archive is read without altering or extracting the preserved file. A complete Friday calendar is reconstructed between the first and last chart dates. Dates absent from the source are recorded as unknown source weeks; they are never coded as song exits.


In [ ]:
chart_archive = ROOT / "data/raw/charts/original/spotify_top_200_weekly_global_2017_2021.zip"
chart = read_chart_archive(chart_archive)
chart["chart_date"] = pd.to_datetime(chart["Week"], errors="raise")
chart["spotify_track_id"] = chart["Link"].map(extract_spotify_id)

observed_weeks = pd.DatetimeIndex(sorted(chart["chart_date"].unique()))
expected_weeks = pd.date_range(observed_weeks.min(), observed_weeks.max(), freq="7D")
missing_source_weeks = expected_weeks.difference(observed_weeks)
duplicate_track_weeks = chart.duplicated(["spotify_track_id", "chart_date"]).sum()

chart_audit = pd.DataFrame([
    ("chart observations", len(chart)),
    ("source columns", chart.shape[1] - 2),
    ("observed weeks", len(observed_weeks)),
    ("unique tracks", chart["spotify_track_id"].nunique()),
    ("minimum rank", chart["Rank"].min()),
    ("maximum rank", chart["Rank"].max()),
    ("missing rank values", chart["Rank"].isna().sum()),
    ("missing stream values", chart["Streams"].isna().sum()),
    ("negative stream values", (chart["Streams"] < 0).sum()),
    ("invalid or missing track IDs", chart["spotify_track_id"].isna().sum()),
    ("duplicate track-week rows", duplicate_track_weeks),
    ("missing source-wide weeks", len(missing_source_weeks)),
], columns=["audit item", "value"])
display(chart_audit)
print("Date coverage:", observed_weeks.min().date(), "to", observed_weeks.max().date())
print("Unknown source weeks:", [d.date().isoformat() for d in missing_source_weeks])

assert len(chart) == 44_200
assert len(observed_weeks) == 221
assert chart["spotify_track_id"].nunique() == 4_247
assert chart["Rank"].between(1, 200).all()
assert chart["Streams"].notna().all() and chart["Streams"].ge(0).all()
assert duplicate_track_weeks == 0
assert [d.date().isoformat() for d in missing_source_weeks] == [
    "2017-06-02", "2017-06-09", "2021-02-19"
]


The chart file contains complete ranks and non-negative weekly streams for every observed track-week. Its three missing weeks are a property of the source calendar, not evidence that every song left and re-entered the chart.

## Feature-source audit

The 817 MB songs file was previously scanned in chunks by the validated raw-data audit. Repeating that full scan here would add execution time without changing the audit; the notebook therefore verifies its header directly and loads the saved row, duplicate and missingness results. The smaller 160k ID column is read directly below.


In [ ]:
raw_audit = load_json(ROOT / "data/processed/audit/raw_dataset_audit.json")
feature_rows = []
for key, label in [
    ("spotify_550k_songs", "550k songs"),
    ("spotify_550k_artists", "550k artists"),
    ("spotify_160k_tracks", "160k tracks"),
]:
    item = raw_audit[key]
    feature_rows.append({
        "source": label,
        "rows (validated audit)": item["rows"],
        "columns": item["column_count"],
        "unique nonblank IDs": item["unique_nonblank_ids"],
        "duplicate IDs beyond first": item["duplicate_id_values_beyond_first"],
        "full duplicate rows": item["duplicate_rows_beyond_first"],
    })
feature_audit = pd.DataFrame(feature_rows)
display(feature_audit)

songs_header = pd.read_csv(ROOT / "data/raw/spotify_550k/songs.csv", nrows=3)
tracks_160_ids = pd.read_csv(ROOT / "data/raw/spotify_160k/data.csv", usecols=["id"])
assert list(songs_header.columns) == raw_audit["spotify_550k_songs"]["columns"]
assert len(tracks_160_ids) == 170_653
assert tracks_160_ids["id"].is_unique

selected_missing = pd.DataFrame([
    ("550k song name", raw_audit["spotify_550k_songs"]["missing_values"]["name"]),
    ("550k album name", raw_audit["spotify_550k_songs"]["missing_values"]["album_name"]),
    ("550k song ID", raw_audit["spotify_550k_songs"]["missing_values"]["id"]),
    ("160k track ID", raw_audit["spotify_160k_tracks"]["missing_values"]["id"]),
], columns=["field", "missing values"])
display(selected_missing)


The three missing song names and twenty missing album names in the full 550k source are documented rather than silently deleted. They do not create missing IDs. The modelling cohorts later contain complete frozen predictor fields; suspicious values remain traceable through the validation logs.

## Exact-ID linkage and coverage

Spotify track ID is the only matching key used. No fuzzy title/artist match enters the analysis. The matched 550k bridge was created by the validated streaming matcher, while the 160k overlap can be checked directly from its unique ID column.


In [ ]:
chart_ids = set(chart["spotify_track_id"].dropna())
matched_550 = pd.read_csv(
    ROOT / "data/processed/chart_survival/matched_550k_baseline_features.csv",
    usecols=["spotify_track_id"],
)
ids_550 = set(matched_550["spotify_track_id"])
ids_160 = set(tracks_160_ids["id"].dropna())
match_summary = pd.DataFrame([
    ("all chart tracks", len(chart_ids), 100.0),
    ("exact 550k matches", len(chart_ids & ids_550), 100 * len(chart_ids & ids_550) / len(chart_ids)),
    ("exact 160k matches", len(chart_ids & ids_160), 100 * len(chart_ids & ids_160) / len(chart_ids)),
    ("exact matches to both feature sources", len(chart_ids & ids_550 & ids_160), 100 * len(chart_ids & ids_550 & ids_160) / len(chart_ids)),
], columns=["linkage population", "tracks", "percent of chart tracks"])
display(match_summary.round({"percent of chart tracks": 2}))
assert len(chart_ids & ids_550) == 1_534
assert len(chart_ids & ids_160) == 1_981
assert len(chart_ids & ids_550 & ids_160) == 969

coverage_by_year = pd.read_csv(
    ROOT / "data/processed/fixed_window_count_v1/feature_coverage_by_chart_entry_year.csv"
)
display(coverage_by_year[[
    "entry_year", "all_chart_tracks", "550k_exact_matches",
    "550k_coverage_percent", "160k_exact_matches", "160k_coverage_percent"
]].round(2))


Exact-ID coverage changes across entry years, and the 550k-linked population is not interchangeable with the full chart population. The main lyric analysis therefore describes the linked cohort; the 160k cohort is retained as a common-feature robustness population.

## Sequential cohort exclusions

Exclusions are mutually exclusive only after applying the documented priority: uncertain entry, insufficient follow-up, missing-source-week contamination, missing feature match, then inclusion. Raw diagnostic flags can overlap and should not be summed.


In [ ]:
cohort_summary = pd.read_csv(ROOT / "data/processed/pre_model_analysis_ready_v1/cohort_summary.csv")
sequential = cohort_summary[
    cohort_summary["count_type"].eq("mutually_exclusive_sequential_status")
    & cohort_summary["cohort"].isin(["550k", "160k"])
].copy()
pivot = sequential.pivot_table(
    index=["horizon_weeks", "reason_or_status"],
    columns="cohort", values="tracks", aggfunc="first"
).reset_index().fillna(0)
display(pivot)

expected_final = {(52, "550k"): 1014, (26, "550k"): 1220, (52, "160k"): 1216, (26, "160k"): 1551}
for (horizon, cohort), expected in expected_final.items():
    rows = sequential[(sequential.horizon_weeks.eq(horizon)) & (sequential.cohort.eq(cohort))]
    assert int(rows.tracks.sum()) == 4_247
    included = int(rows.loc[rows.reason_or_status.eq("included_analysis_ready"), "tracks"].iloc[0])
    assert included == expected

for filename, expected in [
    ("analysis_ready_550k_52w.csv", 1014),
    ("analysis_ready_550k_26w.csv", 1220),
    ("analysis_ready_160k_52w.csv", 1216),
    ("analysis_ready_160k_26w.csv", 1551),
]:
    cohort = pd.read_csv(ROOT / "data/processed/pre_model_analysis_ready_v1" / filename)
    assert len(cohort) == expected
    assert_unique_tracks(cohort, filename)


## Artist and version grouping

Track ID remains the analysis unit. Validation groups use the first credited Spotify artist and are strengthened so supported exact-lyric/version groups cannot cross folds. This does not guarantee separation of every collaborator: the all-collaborator graph contains one component covering most tracks, so that stricter rule would make five balanced folds impractical.


In [ ]:
collaboration = pd.read_csv(
    ROOT / "data/processed/final_pre_model_specification_v1/collaboration_grouping_diagnostic.csv"
)
version_summary = pd.read_csv(
    ROOT / "data/processed/final_pre_model_specification_v1/analysis_unit_version_summary.csv"
)
assignments = pd.read_csv(
    ROOT / "data/processed/final_pre_model_specification_v1/provisional_grouped_cv_assignments.csv"
)
grouping_display = collaboration[
    collaboration["metric"].isin([
        "single_artist_tracks", "multi_artist_tracks", "distinct_first_credited_artists",
        "distinct_all_credited_artists", "connected_components", "largest_component_tracks",
        "largest_component_percent", "group_components", "collaborators_crossing_folds"
    ])
][["section", "metric", "value", "note"]]
display(grouping_display)
display(version_summary.head(12))

protected_flags = [
    "first_artist_crosses_folds", "exact_lyric_group_crosses_folds",
    "related_version_group_crosses_folds"
]
assert len(assignments) == 1014 and assignments.spotify_track_id.is_unique
assert assignments[protected_flags].sum().sum() == 0
assert assignments.primary_group_component.nunique() == 276
assert assignments.fold.value_counts().sort_index().tolist() == [203, 203, 203, 203, 202]


## Exact bridge from strict survival to the final Y52 cohort

The two cohorts answer different questions, so the numerical change is not a one-way sequence of 343 exclusions. The ID bridge below separates removals from the 1,357-track survival cohort and valid fixed-window additions from outside it.


In [ ]:
transition = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/survival1357_to_y52_1014_reconciliation.csv"
)
transition_summary = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/survival1357_to_y52_1014_summary.csv"
)
display(transition_summary)
display(transition[transition.transition_status.ne("retained_from_survival_in_final_y52")][[
    "track_id", "in_survival_1357", "in_final_y52_1014",
    "exact_exclusion_or_addition_reason", "survival_first_observed_entry",
    "chart_entry_date", "source_gap_dates",
]].head(12))

assert transition.in_survival_1357.sum() == 1357
assert transition.in_final_y52_1014.sum() == 1014
assert len(transition.query("in_survival_1357 == 1 and in_final_y52_1014 == 0")) == 353
assert len(transition.query("in_survival_1357 == 0 and in_final_y52_1014 == 1")) == 10
assert 1357 - 198 - 75 - 80 + 10 == 1014


The transition removes 198 tracks without 52 weeks of follow-up, 75 whose required window crosses the two June 2017 source gaps, and 80 whose window crosses the February 2021 gap. Ten tracks are then added because their 52-week fixed window is complete even though their later first spell crosses a source gap and is therefore absent from strict survival. The net reduction is 343. Separately, the earlier 1,025 count became 1,014 after 11 entries immediately following the June gap were classified as uncertain.


## Notebook 01 conclusions

The preserved chart archive contains 44,200 observations, 221 observed weeks and 4,247 identifiable tracks. Exact-ID linkage yields 1,534 matches to the 550k source and 1,981 to the 160k source. After entry, follow-up, missing-week and feature-linkage exclusions, the clean cohorts contain 1,014/1,220 tracks for the 550k 52/26-week analyses and 1,216/1,551 for the corresponding 160k analyses.

Downstream notebooks use `analysis_ready_550k_52w.csv` as the primary feature cohort, `analysis_ready_550k_26w.csv` as a horizon sensitivity, and the two 160k files only for common-feature robustness. Frozen fold assignments are taken from `provisional_grouped_cv_assignments.csv`.


In [ ]:
record_values(RECORD_DIR / "nb01_values.json", "01_data_sources_audit_and_cohort", {
    "chart_rows": len(chart),
    "chart_weeks": len(observed_weeks),
    "chart_tracks": len(chart_ids),
    "missing_source_weeks": len(missing_source_weeks),
    "match_550k": len(chart_ids & ids_550),
    "match_160k": len(chart_ids & ids_160),
    "cohort_550k_52": 1014,
    "cohort_550k_26": 1220,
    "cohort_160k_52": 1216,
    "cohort_160k_26": 1551,
    "protected_components": assignments.primary_group_component.nunique(),
})
print("Notebook 01 reconciliation record saved.")


In [ ]:
record_values(RECORD_DIR / "nb01_supervisor_update_values.json", "01_data_sources_audit_and_cohort", {
    "survival_start_n": int(transition.in_survival_1357.sum()),
    "final_y52_n": int(transition.in_final_y52_1014.sum()),
    "survival_removed_n": int(len(transition.query("in_survival_1357 == 1 and in_final_y52_1014 == 0"))),
    "fixed_window_added_n": int(len(transition.query("in_survival_1357 == 0 and in_final_y52_1014 == 1"))),
    "insufficient_follow_up_removed": int((transition.exact_exclusion_or_addition_reason == "insufficient_52_week_follow_up").sum()),
    "june_gap_removed": int((transition.exact_exclusion_or_addition_reason == "required_52_week_window_crosses_2017_06_source_gaps").sum()),
    "february_gap_removed": int((transition.exact_exclusion_or_addition_reason == "required_52_week_window_crosses_2021_02_19_source_gap").sum()),
})
print("Notebook 01 supervisor-update reconciliation record saved.")
